# Spherical Kappa Analysis

Analysis notebook for **`SphericalKappaField`** — weak-lensing convergence maps on the sphere.

**How to use:**  
Edit the `CONFIGURE` cell below to point to your parquet file(s), then run all cells top-to-bottom.

| Section | Content |
|---|---|
| 0 | Imports & configuration |
| 1 | Load & inspect |
| 2 | Kappa map visualisation |
| 3 | Convergence power spectrum Cℓκ |
| 4 | Theory comparison (weak lensing, Halofit / linear) |
| 5 | Multi-catalog comparison (requires ≥ 2 paths) |

In [ ]:
import os

os.environ["JAX_PLATFORMS"] = "cpu"  # must be before any JAX import

import sys
from pathlib import Path

import jax.numpy as jnp
import jax_cosmo as jc
import jax_fli as jfli
import matplotlib.pyplot as plt
import numpy as np

# Import JCAP style from analysis/utils.py
_here = Path.cwd()
_utils_dir = _here if (_here / "utils.py").exists() else _here / "analysis"
if str(_utils_dir) not in sys.path:
    sys.path.insert(0, str(_utils_dir))
from utils import set_jcap_style

set_jcap_style()

In [ ]:
# ===== CONFIGURE HERE =====

PATHS = [
    "/home/wassim/Projects/NBody/jax-fli/DEL/results/lensing.parquet",
    # Add more paths for multi-catalog comparison (Section 5)
]
LABELS = ["lensing"]

LMIN = 10  # minimum ell for Cℓ plots
LMAX = None  # None → auto (3 * nside - 1)
NONLINEAR_FN = "halofit"  # "halofit" or "linear"
CMAP = "RdBu_r"  # diverging colormap suits signed κ maps

# Fallback source redshifts for theory (used when z_sources not in field metadata)
NZ_SOURCES = [0.5, 1.0, 1.5]

# ===== END CONFIGURATION =====

## 1. Load & Inspect

In [ ]:
catalogs = []
for path, label in zip(PATHS, LABELS):
    cat = jfli.io.Catalog.from_parquet(path)
    fld = cat.field[0]
    cos = cat.cosmology[0]
    catalogs.append({"label": label, "cat": cat, "field": fld, "cosmo": cos})
    print(f"Loaded  : {label}")
    print(f"  path  : {path}")
    print(f"  type  : {type(fld).__name__}")
    print()

kap0 = catalogs[0]["field"]
cos0 = catalogs[0]["cosmo"]

nside = int(kap0.nside)
n_bins = kap0.array.shape[0] if kap0.array.ndim > 1 else 1
lmax_eff = LMAX if LMAX is not None else 3 * nside - 1

print("=== Reference field ===")
print(f"  nside        : {nside}")
print(f"  n_tomo_bins  : {n_bins}")
print(f"  array shape  : {kap0.array.shape}")
print(f"  LMAX used    : {lmax_eff}")

# Extract z_sources for theory comparison
if getattr(kap0, "z_sources", None) is not None:
    z_src_field = list(np.asarray(kap0.z_sources))
    print(f"  z_sources    : {z_src_field}")
    z_src_theory = z_src_field
else:
    z_src_theory = NZ_SOURCES
    print(f"  z_sources    : not in metadata → using NZ_SOURCES = {NZ_SOURCES}")

print(
    f"  κ range      : [{float(kap0.array.min()):.4f}, {float(kap0.array.max()):.4f}]"
)

print("\nCosmology (from parquet):")
for p in ["Omega_c", "Omega_b", "h", "n_s", "sigma8", "w0", "wa"]:
    v = getattr(cos0, p, None)
    if v is not None:
        print(f"  {p:8s} = {float(v):.6g}")

## 2. Kappa Map Visualisation

In [ ]:
# Symmetric colorbar centred on zero (κ can be negative)
kap_arr = np.asarray(kap0.array)
if kap_arr.ndim == 1:
    kap_arr = kap_arr[np.newaxis]

n_plot = n_bins
ncols_m = min(3, n_plot)

fig, axes = kap0.plot(
    ncols=ncols_m,
    cmap=CMAP,
    colorbar=True,
    projection_type="mollweide",
    border_linewidth=0.5,
    show_ticks=False,
)
fig.suptitle(f"{catalogs[0]['label']} — κ maps", y=1.01)
fig.tight_layout()
# fig.savefig("output/spherical_kappa_maps.png", dpi=150, bbox_inches="tight")
plt.show()

## 3. Convergence Power Spectrum Cℓκ

In [ ]:
all_cls = []
for entry in catalogs:
    kap = entry["field"]
    cl = kap.angular_cl(lmax=lmax_eff, method="healpy")
    all_cls.append(cl)
    print(f"{entry['label']}: Cl shape = {np.asarray(cl.array).shape}")

cl0 = all_cls[0]
ell = np.asarray(cl0.wavenumber)  # (n_ell,)
cl0_arr = np.asarray(cl0.array)  # (n_bins, n_ell) or (n_ell,)
if cl0_arr.ndim == 1:
    cl0_arr = cl0_arr[np.newaxis]

mask = ell >= LMIN
ell_m = ell[mask]
cl0_m = cl0_arr[:, mask]

ncols_s = min(3, n_bins)
nrows_s = (n_bins + ncols_s - 1) // ncols_s

fig, axes = plt.subplots(
    nrows_s, ncols_s, figsize=(5 * ncols_s, 3.5 * nrows_s), squeeze=False
)
for i in range(n_bins):
    ax = axes[i // ncols_s, i % ncols_s]
    ax.loglog(ell_m, cl0_m[i], color="tab:blue", label=catalogs[0]["label"])
    ax.grid(True, which="both", ls="--", alpha=0.3)
    ax.set_xlabel(r"$\ell$")
    ax.set_ylabel(r"$C_\ell^{\kappa\kappa}$")
    zs_str = f"  z={z_src_theory[i]:.3f}" if i < len(z_src_theory) else ""
    ax.set_title(f"Bin {i}{zs_str}", fontsize=10)
    ax.legend(fontsize=8)

for j in range(n_bins, nrows_s * ncols_s):
    axes[j // ncols_s, j % ncols_s].set_visible(False)

fig.suptitle(f"Convergence Power Spectrum — {catalogs[0]['label']}", y=1.01)
fig.tight_layout()
# fig.savefig("output/spherical_kappa_cl.png", dpi=150, bbox_inches="tight")
plt.show()

## 4. Theory Comparison

Theory Cℓκ computed via Limber approximation with the weak-lensing kernel and the
cosmology read directly from the parquet.

In [ ]:
nl_fn = jc.power.halofit if NONLINEAR_FN == "halofit" else jc.power.linear

print(f"Computing theory Cl_kappa [{NONLINEAR_FN}]...")
print(f"  z_sources: {z_src_theory}")

theory_cl = jfli.compute_theory_cl(
    cos0,
    ells=jnp.asarray(ell),
    z_source=z_src_theory,
    probe_type="weak_lensing",
    nonlinear_fn=nl_fn,
)
th_arr = np.asarray(theory_cl.array)  # (n_bins, n_ell) or (n_ell,)
if th_arr.ndim == 1:
    th_arr = th_arr[np.newaxis]
th_m = th_arr[:, mask]
n_theory_bins = th_arr.shape[0]
print(f"Theory Cl shape: {theory_cl.array.shape}")

# --- Overlay: sim + theory ---
n_show = min(n_bins, n_theory_bins)
ncols_s = min(3, n_show)
nrows_s = (n_show + ncols_s - 1) // ncols_s

fig, axes = plt.subplots(
    nrows_s, ncols_s, figsize=(5 * ncols_s, 3.5 * nrows_s), squeeze=False
)
for i in range(n_show):
    ax = axes[i // ncols_s, i % ncols_s]
    ax.loglog(
        ell_m, th_m[i], color="black", ls="--", lw=1.5, label=f"Theory ({NONLINEAR_FN})"
    )
    ax.loglog(ell_m, cl0_m[i], color="tab:blue", lw=1.5, label=catalogs[0]["label"])
    ax.grid(True, which="both", ls="--", alpha=0.3)
    ax.set_xlabel(r"$\ell$")
    ax.set_ylabel(r"$C_\ell^{\kappa\kappa}$")
    zs_str = f"  z={z_src_theory[i]:.3f}" if i < len(z_src_theory) else ""
    ax.set_title(f"Bin {i}{zs_str}", fontsize=10)
    if i == 0:
        ax.legend(fontsize=8)

for j in range(n_show, nrows_s * ncols_s):
    axes[j // ncols_s, j % ncols_s].set_visible(False)

fig.suptitle("Sim vs Theory", y=1.01)
fig.tight_layout()
# fig.savefig("output/spherical_kappa_cl_theory.png", dpi=150, bbox_inches="tight")
plt.show()

In [ ]:
# Ratio Cℓκ / Cℓκ_theory
fig, axes = plt.subplots(
    nrows_s, ncols_s, figsize=(5 * ncols_s, 2.5 * nrows_s), squeeze=False
)
for i in range(n_show):
    ax = axes[i // ncols_s, i % ncols_s]
    ratio = cl0_m[i] / th_m[i]
    ax.semilogx(ell_m, ratio, color="tab:blue", lw=1.5, label=catalogs[0]["label"])
    ax.axhline(1.0, color="black", ls=":", lw=1)
    for pct, alpha in [(0.20, 0.10), (0.10, 0.18), (0.05, 0.30)]:
        ax.fill_between(ell_m, 1 - pct, 1 + pct, alpha=alpha, color="gray")
    ax.set_xlim(ell_m[0], ell_m[-1])
    ax.set_ylim(0.6, 1.4)
    ax.set_xlabel(r"$\ell$")
    ax.set_ylabel(r"$C_\ell^\kappa / C_\ell^{\kappa,\rm th}$")
    zs_str = f"  z={z_src_theory[i]:.3f}" if i < len(z_src_theory) else ""
    ax.set_title(f"Bin {i}{zs_str}", fontsize=10)
    ax.grid(True, which="both", ls="--", alpha=0.3)

for j in range(n_show, nrows_s * ncols_s):
    axes[j // ncols_s, j % ncols_s].set_visible(False)

fig.suptitle(f"Ratio: Simulation / Theory ({NONLINEAR_FN})", y=1.01)
fig.tight_layout()
# fig.savefig("output/spherical_kappa_ratio.png", dpi=150, bbox_inches="tight")
plt.show()

## 5. Multi-catalog Comparison

Add more paths to `PATHS` in the configuration cell to activate this section.

In [ ]:
if len(catalogs) < 2:
    print("Only one catalog loaded — add more paths to PATHS for comparison.")
else:
    colors = ["tab:blue", "tab:orange", "tab:green", "tab:red", "tab:purple"]

    # --- Overlay Cl ---
    fig, axes = plt.subplots(
        nrows_s, ncols_s, figsize=(5 * ncols_s, 3.5 * nrows_s), squeeze=False
    )
    for i in range(n_show):
        ax = axes[i // ncols_s, i % ncols_s]
        for ci, (entry, cl) in enumerate(zip(catalogs, all_cls)):
            cl_arr = np.asarray(cl.array)
            if cl_arr.ndim == 1:
                cl_arr = cl_arr[np.newaxis]
            cl_i = cl_arr[i, mask]
            lbl = entry["label"] + (" [Ref]" if ci == 0 else "")
            ax.loglog(ell_m, cl_i, color=colors[ci % len(colors)], lw=1.5, label=lbl)
        ax.grid(True, which="both", ls="--", alpha=0.3)
        ax.set_xlabel(r"$\ell$")
        ax.set_ylabel(r"$C_\ell^{\kappa\kappa}$")
        ax.set_title(f"Bin {i}", fontsize=10)
        if i == 0:
            ax.legend(fontsize=7)

    for j in range(n_show, nrows_s * ncols_s):
        axes[j // ncols_s, j % ncols_s].set_visible(False)

    fig.suptitle("Multi-catalog Cℓκ overlay", y=1.01)
    fig.tight_layout()
    plt.show()

    # --- Ratio vs first catalog ---
    ref_arr = np.asarray(all_cls[0].array)
    if ref_arr.ndim == 1:
        ref_arr = ref_arr[np.newaxis]

    fig2, axes2 = plt.subplots(
        nrows_s, ncols_s, figsize=(5 * ncols_s, 2.5 * nrows_s), squeeze=False
    )
    for i in range(n_show):
        ax = axes2[i // ncols_s, i % ncols_s]
        ref_i = ref_arr[i, mask]
        for ci, (entry, cl) in enumerate(zip(catalogs[1:], all_cls[1:]), 1):
            cl_arr = np.asarray(cl.array)
            if cl_arr.ndim == 1:
                cl_arr = cl_arr[np.newaxis]
            ratio = cl_arr[i, mask] / ref_i
            ax.semilogx(
                ell_m,
                ratio,
                color=colors[ci % len(colors)],
                lw=1.5,
                label=entry["label"],
            )
        ax.axhline(1.0, color="black", ls=":", lw=1)
        for pct, alpha in [(0.10, 0.18), (0.05, 0.30)]:
            ax.fill_between(ell_m, 1 - pct, 1 + pct, alpha=alpha, color="gray")
        ax.set_xlim(ell_m[0], ell_m[-1])
        ax.set_ylim(0.8, 1.2)
        ax.set_xlabel(r"$\ell$")
        ax.set_ylabel(
            rf"$C_\ell^\kappa / C_\ell^{{\kappa,\mathrm{{{catalogs[0]['label']}}}}}$"
        )
        ax.set_title(f"Bin {i}", fontsize=10)
        ax.grid(True, which="both", ls="--", alpha=0.3)
        if i == 0:
            ax.legend(fontsize=7)

    for j in range(n_show, nrows_s * ncols_s):
        axes2[j // ncols_s, j % ncols_s].set_visible(False)

    fig2.suptitle(f"Ratio vs {catalogs[0]['label']}", y=1.01)
    fig2.tight_layout()
    plt.show()